# StreamMapNet 학습 과정 Walkthrough

`tools/train.py` → `plugin/core/apis/mmdet_train.py`가 하는 일을 **한 단계씩 풀어서** 실행합니다.
각 모듈 경계에서 텐서 모양/값을 찍고 시각화해서, 어디가 어떻게 생겼는지 확인하는 것이 목적입니다.

- 데이터: nuScenes **v1.0-mini** (`plugin/configs/nusc_newsplit_480_60x30_mini.py`). AP는 의미 없고 구조 확인용입니다.
- 7장(추론)만 공식 체크포인트(`nusc_newsplit_480_60x30_24e`, 24 epoch, 8 GPU)를 불러와 "학습된 모델의 중간 결과"를 봅니다.
- 여기서 쓰는 hook(`FeatureRecorder`)은 성능 병목 분석 노트북(02)에서 그대로 재사용합니다.

## 전체 데이터 흐름

```
6 cams  img (B, 6, 3, 480, 800)
 │  GridMask (train only, p=0.7)                         backbones/bevformer/grid_mask.py
 ▼
ResNet50-DCN (caffe)  → C3, C4, C5 (stride 8/16/32)      mmdet ResNet
 ▼
FPN  → 3 levels × 256ch                                  mmdet FPN
 ▼
BEVFormer encoder (1 layer)                              backbones/bevformer_backbone.py
   TemporalSelfAttention  (prev_bev=None → 사실상 BEV self-attn)
   SpatialCrossAttention  (BEV pillar 4점 → 6 cam 투영 → deformable attn)
   → BEV feature (B, 256, 50, 100)   # 60m × 30m, 0.6m/cell
 ▼
[Streaming BEV] 이전 BEV를 ego-pose로 warp + ConvGRU 융합   mapers/StreamMapNet.py::update_bev_feature, necks/gru.py
 ▼
MapDetectorHead                                          heads/MapDetectorHead.py
   input_proj 256→512 + sine pos-embed
   query: 100 learnable + (이전 frame top-33 query, MotionMLP로 pose 보정)  → decoder layer 1에서 교체
   6 × [self-attn → deformable cross-attn(BEV, 20 ref pts) → FFN] , layer마다 cls/reg 예측  transformer_utils/MapTransformer.py
 ▼
Loss: layer별 Hungarian matching → Focal(cls) + L1(20점 polyline, 순열 고려) + trans_loss(propagation)
 ▼
memory update: top-33 query / line / matched GT → 다음 frame으로
```

## 0. 환경 설정

In [ ]:
%matplotlib inline
import os, sys, time, copy
sys.path.insert(0, os.path.abspath('.'))   # 노트북 폴더(nb_utils.py)
from nb_utils import *
setup_env()   # cwd=/workspace/StreamMapNet, plugin 모듈 registry 등록

import numpy as np, torch
import matplotlib.pyplot as plt
from mmcv import Config

CFG = 'plugin/configs/nusc_newsplit_480_60x30_mini.py'
# 서버 이전 시 이 두 경로만 바꾸면 됩니다 (config 안의 data_root / out_root도 함께 수정)
OFFICIAL_CKPT = os.environ.get('SMN_CKPT', '/nas2/Rotem_dataset/taeu_outputs/StreamMapNet_outputs/ckpts/nusc_newsplit_480_60x30_24e_official.pth')

cfg = Config.fromfile(CFG)
cfg.data.workers_per_gpu = 2
torch.manual_seed(0); np.random.seed(0)
print(torch.cuda.get_device_name(0))

## 1. Config 핵심 값

| 항목 | 값 | 의미 |
|---|---|---|
| `roi_size` | (60, 30) | BEV 범위 x(전방) 60m, y(좌우) 30m |
| `bev_h, bev_w` | 50, 100 | BEV grid → 셀 하나 0.6m |
| `num_queries` / `topk` | 100 / 33 | decoder query 수, 다음 frame으로 넘기는 query 수 |
| `num_points` | 20 | polyline 하나 = 20점 |
| `num_epochs_single_frame` | 전체 epoch의 1/6 | 처음엔 single-frame으로 학습 → 이후 sequential(streaming) |
| `img_backbone lr_mult` | 0.1 | 이미지 백본은 10배 작은 lr |

In [ ]:
for k in ['roi_size', 'bev_h', 'bev_w', 'num_queries', 'num_points', 'batch_size',
          'num_iters_per_epoch', 'num_epochs', 'num_epochs_single_frame', 'total_iters']:
    print(f'{k:<25s}', cfg[k])
print('sampler      ', cfg.data.shuffler_sampler)
print('optimizer    ', cfg.optimizer)
print('lr_config    ', cfg.lr_config)

## 2. Dataset과 파이프라인

`NuscDataset`(`plugin/datasets/nusc_dataset.py`) 한 샘플 = 6장 이미지 + 지도 벡터(GT).
파이프라인 순서: `VectorizeMap`(지도 polygon → 20점 polyline, [0,1] 정규화, 순열 생성) → 이미지 로드 → PhotoMetricDistortion → resize(480×800, intrinsic도 같이 수정) → normalize → pad → Collect.

In [ ]:
from mmdet3d.datasets import build_dataset
train_ds = build_dataset(cfg.data.train)
print('samples:', len(train_ds), '| scenes(groups):', len(set(train_ds.flag)))
for t in train_ds.pipeline.transforms:
    print(' -', type(t).__name__)

sample = train_ds[0]
describe(sample['img'].data, 'img')
meta = sample['img_metas'].data
print('img_metas keys:', list(meta.keys()))
print('scene:', meta['scene_name'], '| ego2img:', np.asarray(meta['ego2img']).shape)
vec = sample['vectors'].data
for label, lines in vec.items():
    print(f'label {label} ({CAT_NAMES[label]}): {len(lines)} instances, each', lines[0].shape if lines else '-')

GT 하나의 shape가 `(38, 20, 2)`인 이유: **순열(permutation) 후보**입니다 (`pipelines/vectorize.py::permute_line`).
- 닫힌 도형(ped_crossing 등): 시작점 19개 × 방향 2 = 38개 후보
- 열린 선(divider/boundary): 정방향/역방향 2개 + 나머지 36개는 `1e5`로 padding

Loss 계산 시 이 38개 중 예측과 가장 가까운 것과 비교하므로 "점 순서"에 대한 모호성이 사라집니다.

In [ ]:
show_cams(sample['img'].data, cfg.img_norm_cfg, title=f"{meta['scene_name']}  (after resize/normalize)")
fig, ax = plt.subplots(1, 1, figsize=(8, 4))
plot_gt_vectors(ax, vec, cfg.roi_size, title='GT vectors (normalized, ego at center)')
plt.show()

line = [l for l in vec[2]][0] if len(vec[2]) else [l for l in vec[1]][0]
print('open polyline permutations: valid =', int((np.abs(line).max((1, 2)) < 1e4).sum()), '/ 38')

## 3. Sampler: single-frame → streaming 두 단계 학습

`InfiniteGroupEachSampleInBatchSampler`(`datasets/samplers/group_sampler.py`):
- 배치의 **각 슬롯이 자기 scene을 따로 따라갑니다** (슬롯 0은 scene A의 연속 프레임, 슬롯 1은 scene B…).
- `_iters < num_iters_to_seq` 동안은 매 iteration 랜덤 프레임 1장 → 매번 scene이 바뀌므로 `is_first_frame=True` → **memory 미사용(single-frame 학습)**.
- 그 이후엔 scene을 `seq_split_num`개 sub-sequence로 잘라 **순서대로** 공급 → BEV/query memory가 실제로 작동.

아래는 전환 시점을 3 iteration으로 당겨서 슬롯별 (scene, sample_idx)를 찍어본 것입니다.

In [ ]:
from plugin.datasets.builder import build_dataloader
demo_sampler_cfg = copy.deepcopy(cfg.data.shuffler_sampler); demo_sampler_cfg.num_iters_to_seq = 3
dl_demo = build_dataloader(train_ds, cfg.data.samples_per_gpu, 0, dist=False, shuffle=True, seed=0,
                           shuffler_sampler=demo_sampler_cfg, nonshuffler_sampler=cfg.data.nonshuffler_sampler,
                           runner_type=cfg.runner)
bs_iter = iter(dl_demo.batch_sampler)
for it in range(10):
    idxs = next(bs_iter)
    infos = [train_ds.samples[i] for i in idxs]
    phase = 'single' if it < 3 else 'stream'
    print(f'iter {it:2d} [{phase}]  ' + '  |  '.join(f"slot{s}: {x['scene_name']} #{x['sample_idx']}" for s, x in enumerate(infos)))

## 4. 모델 생성

`tools/train.py`와 동일하게 `build_model` → `init_weights()`. (단일 GPU 비분산이라 SyncBN 변환은 생략)
ResNet50은 `open-mmlab://detectron2/resnet50_caffe` ImageNet 가중치로 초기화됩니다.

In [ ]:
from mmdet3d.models import build_model
from mmcv.parallel import MMDataParallel

def make_model(cfg):
    m = build_model(cfg.model, train_cfg=cfg.get('train_cfg'), test_cfg=cfg.get('test_cfg'))
    m.init_weights()
    return m

model = make_model(cfg).cuda()
parts = {
    'backbone.img_backbone (ResNet50)': model.backbone.img_backbone,
    'backbone.img_neck (FPN)': model.backbone.img_neck,
    'backbone.transformer (BEVFormer enc)': model.backbone.transformer,
    'backbone.bev_embedding/pos': torch.nn.ModuleList([model.backbone.bev_embedding, model.backbone.positional_encoding]),
    'stream_fusion_neck (ConvGRU)': model.stream_fusion_neck,
    'head.transformer (decoder x6)': model.head.transformer,
    'head.cls/reg branches': torch.nn.ModuleList([model.head.cls_branches[0], model.head.reg_branches[0]]),
    'head.query_update (MotionMLP)': model.head.query_update,
    'head.others (proj/query emb)': torch.nn.ModuleList([model.head.input_proj, model.head.query_embedding, model.head.reference_points_embed]),
}
for k, m in parts.items():
    print(f'{k:<40s} {count_params(m):7.2f} M')
print(f'{"TOTAL":<40s} {count_params(model):7.2f} M')

## 5. 한 iteration 해부

`model(**data)` → `StreamMapNet.forward_train`이 호출됩니다. 실제 forward를 그대로 돌리면서 forward hook으로 각 모듈 출력만 가로채서 봅니다.
streaming 동작을 보기 위해 **3장의 데모 sampler(전환 3 iter)**로 같은 scene의 연속 프레임이 들어오는 시점까지 돌립니다.

In [ ]:
dl = build_dataloader(train_ds, cfg.data.samples_per_gpu, cfg.data.workers_per_gpu, dist=False, shuffle=True, seed=0,
                      shuffler_sampler=demo_sampler_cfg, nonshuffler_sampler=cfg.data.nonshuffler_sampler,
                      runner_type=cfg.runner)
data_iter = iter(dl)

bb, head = model.backbone, model.head
rec = FeatureRecorder({
    'img_backbone': bb.img_backbone,
    'img_neck': bb.img_neck,
    'bev_raw': bb,                         # BEVFormerBackbone 출력 = fusion 전 BEV
    'gru': model.stream_fusion_neck,       # 샘플마다 1번씩 호출됨
    'head_input_proj': head.input_proj,
    'decoder': head.transformer,           # (layer별 query, layer별 ref points)
    'query_update': head.query_update,     # 이전 frame query가 있을 때만 호출
})

model.train()
# memory는 어차피 detach되어 저장되므로 관찰용 forward는 no_grad로 (graph를 들고 있으면 16GB에서 OOM)
for it in range(5):   # iter 3부터 streaming → iter 4는 같은 scene의 두 번째 프레임
    batch = next(data_iter)
    data = to_device(batch)
    rec.clear()
    with torch.no_grad():
        loss, log_vars, _ = model(**data)
    print(f"iter {it}: scenes={[m['scene_name'] for m in data['img_metas']]}  total loss={log_vars['total']:.2f}  "
          f"query_update called={len(rec['query_update'])}")

### 5.1 GridMask (데이터 증강)
`extract_img_feat`에서 학습 시 확률 0.7로 이미지에 격자 형태 마스크를 씌웁니다.

In [ ]:
img = data['img']                 # (B, 6, 3, H, W)
B, N, C, H, W = img.shape
masked = bb.grid_mask(img.reshape(B * N, C, H, W))
show_cams(masked[:6], cfg.img_norm_cfg, title='GridMask applied (batch slot 0)')

### 5.2 이미지 백본 (ResNet50-DCN) → FPN
- 6장을 batch 차원으로 펼쳐 `(B·6, 3, 480, 800)`로 넣습니다.
- `out_indices=(1,2,3)` → C3/C4/C5 (stride 8/16/32), 채널 512/1024/2048.
- FPN이 3개 level을 모두 256채널로 맞춥니다. 이 3 level이 BEVFormer SCA의 key/value입니다.

In [ ]:
describe(rec['img_backbone'][0], 'ResNet out')
describe(rec['img_neck'][0], 'FPN out')

fpn = rec['img_neck'][0]
fig, axes = plt.subplots(1, 4, figsize=(18, 3))
axes[0].imshow(denorm_img(img[0, 1], cfg.img_norm_cfg)); axes[0].set_title('CAM_FRONT')
for l in range(3):
    axes[l + 1].imshow(feat_to_rgb(fpn[l][1])); axes[l + 1].set_title(f'FPN level {l}  {tuple(fpn[l].shape[-2:])} (PCA)')
for a in axes: a.axis('off')
plt.show()

### 5.3 BEVFormer encoder: 이미지 → BEV
BEV grid 50×100 셀마다 높이 방향 4점(z ∈ [-3, 5]m, pillar)을 만들고 `ego2img`로 6개 카메라에 투영합니다 (`encoder.py::point_sampling`).
투영이 이미지 안에 떨어지는 카메라에서만 deformable attention으로 feature를 뽑습니다.
아래 왼쪽은 **BEV 셀별로 몇 개의 카메라가 보는지**, 오른쪽은 결과 BEV feature입니다.

In [ ]:
enc = bb.transformer.encoder
ref_3d = enc.get_reference_points(bb.bev_h, bb.bev_w, enc.pc_range[5] - enc.pc_range[2], enc.num_points_in_pillar,
                                  dim='3d', bs=B, device=img.device, dtype=torch.float)
ref_cam, bev_mask = enc.point_sampling(ref_3d, enc.pc_range, data['img_metas'])
print('reference_points_cam', tuple(ref_cam.shape), '(num_cam, B, H*W, pillar_pts, uv)')
print('bev_mask            ', tuple(bev_mask.shape))
n_cam_visible = bev_mask[:, 0].any(-1).sum(0).reshape(bb.bev_h, bb.bev_w).cpu()

bev_raw = rec['bev_raw'][0]
describe(bev_raw, 'BEV (before fusion)')
fig, axes = plt.subplots(1, 3, figsize=(18, 3.5))
im = axes[0].imshow(n_cam_visible, extent=[0, 1, 0, 1], origin='upper', cmap='viridis'); plt.colorbar(im, ax=axes[0])
axes[0].set_title('# cameras seeing each BEV cell')
axes[1].imshow(feat_to_rgb(bev_raw[0]), extent=[0, 1, 0, 1], origin='upper'); axes[1].set_title('BEV feature (PCA) - untrained')
plot_gt_vectors(axes[2], data['vectors'][0], cfg.roi_size)
plt.show()

### 5.4 Streaming BEV fusion (`StreamMapNet.update_bev_feature`)
슬롯별로:
- **첫 프레임**: `ConvGRU(h=현재 BEV(detach), x=현재 BEV)`
- **연속 프레임**: 이전 fused BEV를 `prev_g2e @ curr_e2g`로 현재 좌표계에 `grid_sample` warp → `ConvGRU(h=warped, x=현재)`
- 결과는 `bev_memory`에 `detach`되어 저장 → **BPTT 없음** (gradient는 한 프레임 안에서만 흐름)

ConvGRU는 1×1 conv라 **공간적으로 섞지 않고 셀 단위로만** 과거/현재를 섞습니다.

In [ ]:
for i, m in enumerate(data['img_metas']):
    print(f"slot {i}: scene={m['scene_name']}")
print('GRU calls this iter:', len(rec['gru']), '→ shape', tuple(rec['gru'][0].shape))
fused = torch.stack(rec['gru'])          # (B, 256, 50, 100)
mem = model.bev_memory.train_memory_list  # 이번 iter 결과가 저장된 memory
fig, axes = plt.subplots(B, 2, figsize=(12, 2.8 * B))
for i in range(B):
    axes[i, 0].imshow(feat_to_rgb(bev_raw[i]), extent=[0, 1, 0, 1]); axes[i, 0].set_title(f'slot {i} BEV before GRU')
    axes[i, 1].imshow(feat_to_rgb(fused[i]), extent=[0, 1, 0, 1]); axes[i, 1].set_title(f'slot {i} BEV after GRU')
plt.tight_layout(); plt.show()

### 5.5 Head 입력 준비와 query propagation (`MapDetectorHead.propagate`)
1. `input_proj`: BEV 256ch → 512ch, + sine positional embedding
2. learnable query 100개 → `reference_points_embed`로 초기 ref points `(100, 20, 2)` 생성
3. 이전 frame이 있는 슬롯: memory의 top-33 query에 **pose 변화(3×4 행렬 12개 값)**를 `MotionMLP`로 주입, ref points는 좌표변환
4. decoder **layer 1**(`prop_add_stage=1`)에서 현재 query 중 점수 상위 67개 + propagated 33개로 교체

In [ ]:
describe(rec['head_input_proj'][0], 'input_proj out (BEV 512ch)')
if rec['query_update']:
    describe(rec['query_update'][0], 'MotionMLP out (propagated queries)')
else:
    print('이번 iteration은 모든 슬롯이 첫 프레임 → propagation 없음')
inter_queries, init_ref, inter_refs = rec['decoder'][0]
describe(inter_queries, 'decoder queries per layer')
describe(inter_refs, 'decoder ref points per layer')

### 5.6 Decoder 6 layer: layer마다 polyline이 어떻게 바뀌는지
각 layer 출력 query로 `reg_branch`가 20점 좌표를 **절대 좌표로 다시 예측**합니다 (`predict_refine=False` → 이전 layer에 offset을 더하는 방식이 아님).
아직 학습 전이라 랜덤에 가깝습니다. 7장에서 학습된 모델로 같은 그림을 다시 봅니다.

In [ ]:
def plot_layers(inter_refs, inter_queries, model, slot=0, thr=0.3, gt=None, roi=cfg.roi_size):
    L = len(inter_refs)
    fig, axes = plt.subplots(1, L + (gt is not None), figsize=(4 * (L + 1), 2.6))
    for l in range(L):
        scores = model.head.cls_branches[l](inter_queries[l][slot])
        plot_pred_lines(axes[l], inter_refs[l][slot], scores, roi, thr=thr, title=f'layer {l}')
    if gt is not None:
        plot_gt_vectors(axes[-1], gt, roi)
    plt.tight_layout(); plt.show()

gt0 = {k: [np.asarray(x) for x in v] for k, v in data['vectors'][0].items()}
with torch.no_grad():
    plot_layers(inter_refs, inter_queries, model, slot=0, thr=0.05, gt=gt0)

### 5.7 Loss
layer마다 독립적으로 Hungarian matching(`assigner/assigner.py`, cost = Focal cls 5.0 + L1 lines 50.0, 순열 중 최소) 후
- `cls`: FocalLoss (weight 5.0)
- `reg`: LinesL1Loss (weight 50.0)
- `trans_loss`: propagated query를 `reg_branch`에 통과시킨 결과 vs **이전 frame에서 매칭된 GT를 현재 좌표로 변환한 것** (weight 0.1). MotionMLP가 pose 보정을 제대로 하는지 학습시키는 항.

In [ ]:
import pandas as pd
# log key: d{i}.cls / d{i}.reg = decoder layer i, prefix 없는 cls / reg = 마지막 layer(5)
rows = {}
for k, v in log_vars.items():
    if k in ('cls', 'reg'):
        rows.setdefault(k, {})['layer5'] = v
    elif k.startswith('d') and '.' in k:
        layer, name = k.split('.')
        rows.setdefault(name, {})['layer' + layer[1:]] = v
pd.set_option('display.precision', 3)
display(pd.DataFrame(rows))
print({k: round(v, 4) for k, v in log_vars.items() if k in ('trans_loss', 'total')})

### 5.8 Backward와 gradient 흐름
모듈별 gradient norm을 보면 어떤 모듈이 loss에 얼마나 민감한지 감이 옵니다. (memory는 detach되므로 이전 frame으로는 gradient가 가지 않습니다.)

In [ ]:
from mmcv.runner import build_optimizer
optimizer = build_optimizer(model, cfg.optimizer)
print('param groups:', len(optimizer.param_groups), '| lrs:', sorted({g['lr'] for g in optimizer.param_groups}))

model.train(); optimizer.zero_grad()
batch = next(data_iter); data = to_device(batch)
loss, log_vars, _ = model(**data)
loss.backward()

def grad_norm(m):
    g = [p.grad.norm() ** 2 for p in m.parameters() if p.grad is not None]
    return float(torch.stack(g).sum().sqrt()) if g else 0.0
for k, m in parts.items():
    print(f'{k:<40s} grad norm = {grad_norm(m):9.4f}')
total = torch.nn.utils.clip_grad_norm_(model.parameters(), **cfg.optimizer_config.grad_clip)
print('total grad norm before clip (max 35):', float(total))
optimizer.step()

## 6. 짧은 학습 루프

`IterBasedRunner`가 하는 일을 직접 씁니다: `train_step` → `backward` → grad clip(35) → `step`.
(warmup + cosine lr 스케줄은 생략) mini 데이터로 수십 iteration만 돌려 loss가 내려가는지만 확인합니다.

In [ ]:
rec.remove()   # 관찰용 hook 제거 (붙여두면 매 iter 출력이 쌓여 OOM)
del loss; torch.cuda.empty_cache(); torch.cuda.reset_peak_memory_stats()

N_ITERS = 60
model.train()
hist = []
t0 = time.time()
for it in range(N_ITERS):
    data = to_device(next(data_iter))
    optimizer.zero_grad()
    loss, log_vars, _ = model(**data)
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), **cfg.optimizer_config.grad_clip)
    optimizer.step()
    hist.append({k: log_vars[k] for k in ('cls', 'reg', 'trans_loss', 'total')})
dt = (time.time() - t0) / N_ITERS
print(f'{dt:.2f} s/iter (bs={cfg.batch_size}),  peak mem {torch.cuda.max_memory_allocated() / 2**30:.1f} GB')

h = pd.DataFrame(hist)
h.rolling(5, min_periods=1).mean().plot(subplots=True, figsize=(10, 6), layout=(-1, 2)); plt.tight_layout(); plt.show()

## 7. 추론과 평가 (공식 체크포인트)

`forward_test`는 bs=1, **val 순서대로**(scene 단위 연속) 돌며 memory를 이어갑니다.
마지막 decoder layer 출력만 `post_process`로 결과에 씁니다. 평가는 Chamfer distance 기반 AP (임계값 0.5/1.0/1.5m 평균).

> 주의: mini val은 2개 scene뿐이라 AP는 참고용입니다. 실제 기준값은 전체 val로 구해야 합니다(02 노트북).

In [ ]:
from mmcv.runner import load_checkpoint
del model, optimizer; torch.cuda.empty_cache()   # 16GB GPU: 학습용 모델 해제
eval_model = make_model(cfg)
ckpt = load_checkpoint(eval_model, OFFICIAL_CKPT, map_location='cpu')
eval_model.cuda()
eval_model.eval()   # StreamMapNet.eval()은 self를 반환하지 않으므로 체이닝 금지
print('checkpoint iter:', ckpt['meta']['iter'])

val_ds = build_dataset(cfg.data.val)
val_ds.work_dir = os.path.join(cfg.out_root, 'nb01_eval')
val_dl = build_dataloader(val_ds, 1, 2, dist=False, shuffle=False,
                          shuffler_sampler=cfg.data.shuffler_sampler, nonshuffler_sampler=cfg.data.nonshuffler_sampler)
print('val samples:', len(val_ds))

rec_eval = FeatureRecorder({'bev_raw': eval_model.backbone, 'gru': eval_model.stream_fusion_neck,
                            'decoder': eval_model.head.transformer})
eval_dp = MMDataParallel(eval_model, device_ids=[0])
results, keep = [], {}
with torch.no_grad():
    for i, batch in enumerate(val_dl):
        rec_eval.clear()
        res = eval_dp(return_loss=False, rescale=True, **batch)
        results.extend(res)
        if i in (0, 10):   # scene 첫 프레임 / 11번째 프레임 저장
            keep[i] = dict(batch=batch, bev=rec_eval['bev_raw'][0], fused=rec_eval['gru'][0],
                           dec=rec_eval['decoder'][0])
rec_eval.remove()
print('results:', len(results), '| keys:', list(results[0].keys()))

In [ ]:
for i, k in keep.items():
    # val 파이프라인엔 GT가 없으므로 train 파이프라인의 VectorizeMap으로 GT 생성
    s = train_ds.pipeline.transforms[0](copy.deepcopy(val_ds.get_sample(i)))
    gt = {lab: [l for l in ls] for lab, ls in s['vectors'].items()}
    fig, axes = plt.subplots(1, 2, figsize=(10, 2.6))
    axes[0].imshow(feat_to_rgb(k['bev'][0]), extent=[0, 1, 0, 1]); axes[0].set_title(f'frame {i}: BEV before GRU')
    axes[1].imshow(feat_to_rgb(k['fused']), extent=[0, 1, 0, 1]); axes[1].set_title(f'frame {i}: BEV after GRU')
    plt.show()
    q, _, refs = k['dec']
    with torch.no_grad():
        plot_layers(refs, q, eval_model, slot=0, thr=0.4, gt=gt)

In [ ]:
metrics = val_ds.evaluate(results)
metrics

## 8. 정식 학습 실행 방법

노트북 루프는 lr 스케줄/체크포인트/평가 hook이 빠져 있습니다. 실제 학습은:

```bash
cd /workspace/StreamMapNet
# 단일 GPU라도 config가 SyncBN=True 이므로 분산 launcher(1 proc)로 실행
bash tools/dist_train.sh plugin/configs/nusc_newsplit_480_60x30_mini.py 1
# 평가
bash tools/dist_test.sh plugin/configs/nusc_newsplit_480_60x30_mini.py <ckpt> 1 --eval
```